In [ ]:
!pip install -q rasterio  

In [ ]:
# === PRE-TRAITEMENT NDVI BRUT (1 band) ============================
import numpy as np, glob, rasterio, os, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

data_dir = "put the data link here"
seq_len  = N #choose the sequence length

files = sorted(glob.glob(os.path.join(data_dir, "*.tif")))
print("✅", len(files), "images founded")
if not files:
    raise RuntimeError("No GeoTIFF NDVI founded")

# --------- load NDVI (1 band) ------------------------------
def read_ndvi(path):
    with rasterio.open(path) as src:
        arr = src.read(1).astype(np.float32)
    arr = np.nan_to_num(arr, nan=0.0)
    
    arr = (arr + 1.0) / 2.0
    return np.clip(arr, 0, 1)            # (H, W)

stack = np.stack([read_ndvi(p) for p in files])   # (N, H, W)
H, W   = stack.shape[1:]

# --------- sequences  ----------------------------------
X, y = [], []
for t in range(len(stack) - seq_len):
    X.append(stack[t:t+seq_len])   # (N, H, W)
    y.append(stack[t+seq_len])     # (H, W)

X = np.array(X)[..., np.newaxis]   # (n, N, H, W, 1)
y = np.array(y)[..., np.newaxis]   # (n, H, W, 1)

print("X :", X.shape, "| y :", y.shape)

# --------- train / test -------------------------------------------
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
print("Train :", Xtr.shape, "| Test :", Xte.shape)

os.makedirs("/data", exist_ok=True)
np.save("X_train.npy", Xtr)
np.save("y_train.npy", ytr)
np.save("X_test.npy",  Xte)
np.save("y_test.npy",  yte)

# --------- aperçu --------------------------------------------------
plt.figure(figsize=(8,3.5))
plt.subplot(1,2,1); plt.imshow(Xtr[0,-1,:,:,0], cmap="RdYlGn"); plt.title("t-1"); plt.axis("off")
plt.subplot(1,2,2); plt.imshow(ytr[0,:,:,0],   cmap="RdYlGn"); plt.title("t");   plt.axis("off")
plt.tight_layout(); plt.show()


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model
import numpy as np

H, W = 372, 743
N = H * W

# --- 1. Adjacency Sparse ---
def create_grid_adjacency_sparse(h, w):
    row_idx, col_idx = [], []
    for r in range(h):
        for c in range(w):
            i = r * w + c
            for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                rr, cc = r + dr, c + dc
                if 0 <= rr < h and 0 <= cc < w:
                    j = rr * w + cc
                    row_idx.append(i)
                    col_idx.append(j)
    indices = np.stack([row_idx, col_idx], axis=1)
    values = np.ones(len(row_idx), dtype=np.float32)
    A_sparse = tf.sparse.SparseTensor(indices=indices, values=values, dense_shape=[N, N])
    return tf.sparse.reorder(A_sparse)

A_sparse = create_grid_adjacency_sparse(H, W)

# --- 2. Layer GCN ---
class GraphConvLayer(layers.Layer):
    def __init__(self, out_features, **kwargs):
        super().__init__(**kwargs)
        self.out_features = out_features

    def build(self, input_shape):
         F_in = input_shape[-1]
         self.W = self.add_weight(name="W_gcn", shape=(F_in, self.out_features), initializer='glorot_uniform', trainable=True)
         self.b = self.add_weight(name="b_gcn", shape=(self.out_features,), initializer='zeros', trainable=True)
       

    def call(self, X):
        def apply_graph_conv(Xb):
            AXb = tf.sparse.sparse_dense_matmul(A_sparse, Xb)
            return tf.matmul(AXb, self.W) + self.b
        return tf.map_fn(apply_graph_conv, X)

# --- 3.  Transformer Bloc  ---
class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super().__init__()
        self.att = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential([
            layers.Dense(ff_dim, activation='relu'),
            layers.Dense(embed_dim),
        ])
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(rate)
        self.dropout2 = layers.Dropout(rate)

    def call(self, inputs, training=None):  
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)

        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)


# --- 4. Full Model ---
def build_model_convlstm_gcn_transformer(input_shape=(11, H, W, 1)):
    inputs = layers.Input(shape=input_shape)

    # ConvLSTM
    x = layers.ConvLSTM2D(64, (3, 3), padding='same', return_sequences=False)(inputs)
    x = layers.BatchNormalization()(x)  # (batch, H, W, 64)

    # Reshape to (batch, N, 64)
    x = layers.Reshape((N, 64))(x)

    # GCN
    x = GraphConvLayer(32)(x)  # (batch, N, 32)
    x = layers.Activation('relu')(x)

    # Transformer
    x = TransformerBlock(embed_dim=32, num_heads=4, ff_dim=64)(x)  # (batch, N, 32)

    # Reshape to image
    x = layers.Reshape((H, W, 32))(x)

    # output
    outputs = layers.Conv2D(1, (3, 3), activation='sigmoid', padding='same')(x)

    model = Model(inputs, outputs)
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

# === 5. architecture test ===
if __name__ == "__main__":
    model = build_model_convlstm_gcn_transformer()
    model.summary()


In [ ]:

from tensorflow.keras.callbacks import ModelCheckpoint, CSVLogger,EarlyStopping,ReduceLROnPlateau


X_train = np.load("/data/X_train.npy")
y_train = np.load("/data/y_train.npy")
X_test = np.load("/data/X_test.npy")
y_test = np.load("/data/y_test.npy")

model = build_model_convlstm_gcn_transformer(input_shape=X_train.shape[1:])
checkpoint = ModelCheckpoint("ConvLstm_gcn_trans.keras", save_best_only=True, monitor="val_loss", mode="min")
csv_logger = CSVLogger("training_log.csv", append=True)
ReduceLR=ReduceLROnPlateau(patience=5, factor=0.5, verbose=1),
Earlystop= EarlyStopping(patience=10, restore_best_weights=True)
history = model.fit(X_train, y_train, 
                    validation_data=(X_test, y_test), 
                    epochs=100, batch_size=2,
                    callbacks=[checkpoint, csv_logger,Earlystop,ReduceLR])